# NB_lab_test_trends

Builds `adb_rapollocare.gold.lab_test_trends` (one row per test type per calendar month) from `silver.lab_results` per the Silver-to-Gold STTM, LEFT JOINed to calendar on the first day of the month, with a LAG-based month-over-month abnormal-rate trend. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE OR REPLACE TABLE adb_rapollocare.gold.lab_test_trends
COMMENT 'Gold: one row per test type per calendar month, incl. month-over-month abnormal-rate trend. Built by NB_lab_test_trends (FULL rebuild).'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                         AS year,
    month(test_date)                        AS month,
    count(*)                                AS total_tests,
    count_if(result_status = 'Abnormal')    AS abnormal_test_count,
    count_if(result_status = 'Critical')    AS critical_test_count
  FROM adb_rapollocare.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(round(m.abnormal_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(round(m.critical_test_count * 100.0 / m.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                                  AS year,
  CAST(t.month AS INT)                                 AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                           AS total_tests,
  CAST(t.abnormal_test_count AS INT)                   AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                   AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  t.prev_month_abnormal_rate_pct,
  CASE
    WHEN t.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                                  AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN adb_rapollocare.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (job / dbutils.notebook.run)
updated_df = spark.table("adb_rapollocare.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))